In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if device=="cuda" else torch.float32)
model = model.to(device)

In [ ]:

def generate_with_prompt(system_prompt, user_prompt, max_new_tokens=400, temperature=0.7):
    full_prompt = f"""<|system|>
{system_prompt}</s>
<|user|>
{user_prompt}</s>
<|assistant|>"""
    
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only assistant response
    return response.split("<|assistant|>")[-1].strip()

# Test
print(generate_with_prompt(
    system_prompt="You are a knowledgeable and friendly Ethiopian AI assistant.",
    user_prompt="Tell me about the history of Addis Ababa."
))

In [ ]:
def ethiopian_ai_assistant(query, mode="normal"):
    if mode == "cot":  # Chain of Thought
        system = "You are an expert Ethiopian culture and knowledge assistant. Think step by step before answering."
    else:
        system = "You are a helpful, accurate, and friendly Ethiopian AI assistant."
    
    return generate_with_prompt(system, query, temperature=0.65)

# Test different queries
queries = [
    "Recommend 3 traditional Ethiopian foods for a tourist.",
    "Explain the difference between Orthodox Christianity and Islam in Ethiopia.",
    "Help me write a short CV for a software engineering job."
]

for q in queries:
    print(f"\n👤 Question: {q}")
    print("🤖 Answer:", ethiopian_ai_assistant(q)[:500] + "...")

In [ ]:
print("\n=== Try Different Prompting Techniques ===")